In [1]:
import utils

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
import re
import os
import seaborn as sns # Loads in the theme

plt.rcParams.update(plt.rcParamsDefault)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 1000)

In [2]:
# Preparation
dataset = 'TUS' # TUS, joystick, speakup, africa

# Loads the datapath, subject_id's and session numbers based on whether we're analysing the study or pilot data
if dataset == 'TUS' or dataset == 'joystick':
    raw_output_path = '/Volumes/project/3025011.02/raw'
    unique_subject_ids = [52, 53] #[52, 53, 54, 55, 57]
    unique_sessions = [2, 3, 4]
elif dataset == 'pilot':
    raw_output_path = '/Volumes/project/3025011.02/long_experiment_data/phd_1_task/experiment_output'
    unique_subject_ids = [5]#[5, 10, 14, 15, 16, 20, 22, 23, 25, 28, 29, 37, 38]
    unique_sessions = [2, 3, 4]
elif dataset=='speakup':
    raw_output_path = '/Volumes/project/3025011.01/4kenneth/SpeakUp!2.0/experiment_output'
    unique_subject_ids = [2]#[2, 3, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 25]
    unique_sessions = [3]
elif dataset=='africa':
    raw_output_path = '/Volumes/project/3025011.01/4kenneth/RL_task-speakup_version_martin_southafrica/experiment_output'
    unique_subject_ids = [2]#[2, 3, 4, 5, 6, 8, 9, 10, 12, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28]
    unique_sessions = [1]

# Make output folder
output_dir = os.path.join('..','plots',dataset,'fMRI','volume_based')
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Load and combine the datasets
combined_results_dataframe = utils.import_functions.main(raw_output_path, unique_subject_ids, unique_sessions, dataset)

These participants will be excluded: {'sub-011_session-02', 'sub-011_session-04', 'sub-011_session-03', 'sub-011_session-01', 'sub-016_session-04'}



In [3]:
REPORT_COLUMNS = [
    "stats_image", "n_voxels", "min", "10%", "mean", "median",
    "90%", "max", "stddev", "vox_fmri_space", "mm_standard_space"
]

STAT_COLS = ["n_voxels", "min", "10%", "mean", "median",
             "90%", "max", "stddev"]

fsl_output_folder = '/Volumes/project/3025011.02/bids/derivatives/fsl'


def load_roi_stats_for_subject(sub, fsl_root):
    """
    Find all featquery report.txt files for a given subject by walking every
    second-level .gfeat -> copeN.feat -> featquery_* tree. Flatten to a dict
    of column_name -> value, where the gfeat-folder suffix and ROI name are
    included in the key so stats from different feat variants and ROIs don't
    overwrite each other.

    Path layout assumed:
      {fsl_root}/{sub}/{sub}_second-level_{suffix}.gfeat/
          cope{N}.feat/featquery_{roi}/report.txt
    """
    sub_dir = Path(fsl_root) / sub
    if not sub_dir.is_dir():
        return {}

    # Match e.g. 'sub-052_second-level_a_comp_cor.gfeat' and capture suffix
    gfeat_pattern = re.compile(rf'^{re.escape(sub)}_second-level_(.+)\.gfeat$')
    cope_dir_pattern = re.compile(r'^cope(\d+)\.feat$')

    flat = {}
    for gfeatdir in sorted(sub_dir.iterdir()):
        gm = gfeat_pattern.match(gfeatdir.name)
        if not gm or not gfeatdir.is_dir():
            continue
        feat_suffix = gm.group(1)  # e.g. 'a_comp_cor'

        for cope_feat in sorted(gfeatdir.iterdir()):
            cm = cope_dir_pattern.match(cope_feat.name)
            if not cm or not cope_feat.is_dir():
                continue
            cope_n = int(cm.group(1))

            for fq_dir in sorted(cope_feat.glob('featquery_*')):
                report_path = fq_dir / 'report.txt'
                if not report_path.is_file():
                    continue

                roi = fq_dir.name[len('featquery_'):]

                df = pd.read_csv(
                    report_path,
                    sep=r'\s+', header=None,
                    names=REPORT_COLUMNS, engine='python'
                )

                for _, row in df.iterrows():
                    for stat in STAT_COLS:
                        parts = [feat_suffix, f'cope{cope_n}', roi, stat]
                        col = '_'.join(parts)
                        flat[col] = row[stat]

    return flat


def add_roi_stats_to_combined(combined_df, fsl_root):
    """
    Add second-level ROI stats to combined_df. Because second-level results
    are aggregated across sessions, each subject's stats are broadcast to all
    of that subject's rows (regardless of session).
    """
    cache = {
        sub: load_roi_stats_for_subject(sub, fsl_root)
        for sub in combined_df['subject_id'].unique()
    }

    stats_rows = [cache[sub] for sub in combined_df['subject_id']]
    stats_df = pd.DataFrame(stats_rows, index=combined_df.index)

    missing = [s for s, v in cache.items() if not v]
    if missing:
        print(f'No ROI stats found for {len(missing)} subject(s):')
        for sub in missing:
            print(f'  {sub}')

    return pd.concat([combined_df, stats_df], axis=1)

In [4]:
# --- Run it ---
combined_results_dataframe = add_roi_stats_to_combined(
    combined_results_dataframe, fsl_output_folder
)

TypeError: unsupported operand type(s) for /: 'PosixPath' and 'int'

In [17]:
print(combined_results_dataframe)

     trial  emotional_cue_time  response objectively_correct subjectively_correct  response_time   RT_s  feedback_time  stimuli_type  trial_duration  probability_condition  face_type stimuli_name  reward_amount subject_id session  objectively_correct_boolean  subjectively_correct_boolean  subjectively_correct_boolean_one_back   congruency valence  previous_outcome previous_response  stay response_before_previous_trial  choicestickiness  win_stay  lose_shift volatility stimulation_condition most_rewarding_response  response_boolean  most_rewarding_response_boolean  trials_since_reversal
0        8        1.741947e+09     avoid               False                False   1.741947e+09  0.635   1.741947e+09             1        3.983508                     20         15    Angry_b15           -0.3        005      02                            0                             0                                      0  incongruent   angry                -1          approach    -1                 

In [18]:
combined_results_pivoted = (
    combined_results_dataframe
    .groupby(['subject_id','stimulation_condition'])[['a_comp_cor_cope2_harvard_oxford_amygdala_mean', 'a_comp_cor_cope3_harvard_oxford_amygdala_mean']]
    .mean()
    .reset_index()
)

combined_results_pivoted = combined_results_pivoted[combined_results_pivoted['stimulation_condition']!='dacc']

combined_results_wide = (
    combined_results_pivoted
    .pivot(
        index=['subject_id'],   # whatever identifies a row
        columns='stimulation_condition',                # the column with the two strings
        values=['a_comp_cor_cope5_harvard_oxford_amygdala_L_zstat_mean', 'a_comp_cor_cope5_harvard_oxford_amygdala_R_zstat_mean']
    )
)
combined_results_wide.columns = [f"{val}_{cond}" for val, cond in combined_results_wide.columns]
combined_results_wide = combined_results_wide.reset_index()
print(combined_results_wide)

# Plot the four groups using a raincloud plot
utils.plotting_functions.paired_raincloud_4_groups(
    combined_results_wide,
    columns=['a_comp_cor_cope5_harvard_oxford_amygdala_L_zstat_mean_sham','a_comp_cor_cope5_harvard_oxford_amygdala_L_zstat_mean_amygdala','a_comp_cor_cope5_harvard_oxford_amygdala_R_zstat_mean_sham','a_comp_cor_cope5_harvard_oxford_amygdala_R_zstat_mean_amygdala'],
    labels=['Left amygdala & sham stimulation','Left amygdala & amygdala stimulation','Right amygdala & sham stimulation','Right amygdala & amygdala stimulation'],
    output_dir=output_dir,
    ylabel_name='Z-scored BOLD dose-response',
    plot_name='z-statistic_test'
)

KeyError: "Columns not found: 'a_comp_cor_cope3_harvard_oxford_amygdala_mean', 'a_comp_cor_cope2_harvard_oxford_amygdala_mean'"